# 07 - Generator lineages: whose stories make the best emotion probes?

**Model probed throughout: google/gemma-4-31b-it** (the instruct model). The
generators only write the stories; every vector is extracted from Gemma
reading them.

**The question.** Emotion vectors are built from story corpora. E5/E6 showed
they are corpus-dependent objects, and E10 showed self-generated stories beat
an external corpus written by a much weaker model (Gemma-4-4B). That left two
explanations tangled: does the probed model need its OWN stories
(distribution match), or just GOOD stories (generator quality)? E11 and E12
untangle them with third-party corpora from a strong external generator
(DeepSeek-v4-pro via OpenRouter).

**Index**
1. The lineages, side by side (what each corpus looks like)
2. Lexical diversity: the scaffold-degeneracy covariate
3. Detection accuracy per layer (E11's registered R1 read)
4. Dose-response: how many stories do probes actually need? (E12)
5. Direction geometry across lineages (E11's R2 read)
6. The preference read: where corpus quality still pays (R3)
7. What we are saying, exactly (verdict and caveats)

**Definitions used everywhere.** "Battery": our 12-emotion probe set (happy,
inspired, loving, proud, calm, desperate, angry, guilty, sad, afraid,
nervous, surprised). "Contrast probe": an emotion's mean story activation
minus the mean over all 12, unit-normalized. "Dual-battery bar": a layer
passes if the probes place the target emotion in the top 3 by centered
cosine for at least 8 of 12 scenarios on BOTH the paper battery and a
held-out battery. "RSA" (representational similarity analysis): correlate
the 12x12 emotion-similarity matrices of two probe sets. Evidence files:
`results/e11_lineage.json`, `results/e12_scale_curve_fixed_arm.json`,
`results/e12_diverse_fullcorpus_grid.json`, `results/e12_scale_curve.json`
(all committed; corpora and vectors on HF, see `emotion_vectors.artifacts`).


In [1]:
# this cell loads the frozen evidence JSONs and the three story corpora
import json
from pathlib import Path

import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RES = ROOT / "results"
MAIN_RES = Path("/Users/abo-tresol/Documents/ai-safety/cbai_project/results")
MODEL = "google/gemma-4-31b-it"

def load_json(name):
    for base in (RES, MAIN_RES):
        p = base / name
        if p.exists():
            return json.loads(p.read_text())
    raise FileNotFoundError(name)

e11 = load_json("e11_lineage.json")
fixed_curve = load_json("e12_scale_curve_fixed_arm.json")
full_grid = load_json("e12_diverse_fullcorpus_grid.json")
try:  # both-arm dose-response; written by scripts/score_e12_scale.py
    both_curves = load_json("e12_scale_curve.json")
except FileNotFoundError:
    both_curves = None
    print("e12_scale_curve.json not present yet: section 4 shows the fixed arm only")

def load_grouped(name):
    for base in (RES, MAIN_RES):
        p = base / name / "stories_grouped.jsonl"
        if p.exists():
            return {r["emotion"]: r["stories"] for r in map(json.loads, p.read_text().splitlines())}
    return None

corpora = {
    "fixed DeepSeek (E11)": load_grouped("openrouter_stories"),
    "diverse DeepSeek (E12)": load_grouped("openrouter_stories_diverse"),
}
print({k: (sum(map(len, v.values())) if v else None) for k, v in corpora.items()})


e12_scale_curve.json not present yet: section 4 shows the fixed arm only
{'fixed DeepSeek (E11)': 3070, 'diverse DeepSeek (E12)': 12262}


## 1. The lineages, side by side

Three corpora, one instruction core ("write a short third-person story,
around 150 words, about a person experiencing X, without naming X"):

| lineage | generator | n per emotion | prompt recipe | HF dataset |
|---|---|---|---|---|
| self-generated | gemma-4-31b-it (the probed model itself) | 256 | fixed instruction | emotion-stories-gemma-4-31b-it |
| fixed DeepSeek | deepseek-v4-pro | 256 | identical fixed instruction | emotion-stories-deepseek-v4-pro |
| diverse DeepSeek | deepseek-v4-pro | 1024 | instruction plus a pinned persona x setting from a deterministic 8x8 grid | emotion-stories-deepseek-v4-pro-diverse |

**How to read the samples below:** all three stories answer the same
assignment ("sad"). Notice the self-generated corpus's house style (it
famously names 98% of its protagonists Elias), versus DeepSeek's varied
scenes, versus the diverse arm where the persona and setting were pinned by
us. The samples are the first story of the emotion in each corpus, not
cherry-picked.


In [2]:
# this cell prints one "sad" story from each available corpus
for name, corp in corpora.items():
    if corp:
        print(f"=== {name} | assigned emotion: sad ===")
        print(corp["sad"][0][:500])
        print()


=== fixed DeepSeek (E11) | assigned emotion: sad ===
The kettle began its slow, mournful whistle. He watched his hand reach out, turn off the stove, and pour the steaming water into a mug he’d forgotten to put a tea bag in. Clear water. He stared at it, then left it on the counter.

Rain traced silver threads down the windowpane, each droplet catching the grey afternoon light before sliding out of sight. A book lay open on the arm of his chair, unread since Tuesday. The dust motes hung suspended in the quiet air, undisturbed.

He lowered himself o

=== diverse DeepSeek (E12) | assigned emotion: sad ===
The old man’s stall was a sliver of silver between a towering display of mobile phone cases and a vendor shouting about ripe mangoes. His hands, crosshatched with scars like old netting, rested motionless on his knees. Before him, three mackerel lay on a bed of crushed ice, their iridescent scales catching the harsh fluorescent light of the market.

A river of people surged past, their g

## 2. Lexical diversity: the scaffold-degeneracy covariate

E11's exploratory R4 read: mean pairwise 5-gram Jaccard overlap within each
corpus (higher = stories share more exact 5-word phrases). **What this
buys:** it separates "the corpus matches the model's distribution" from "the
corpus collapsed onto one scaffold". The self-generated corpus is 53x more
self-similar than fixed DeepSeek, yet its probes work, so scaffold
degeneracy is not what makes probes function. The diverse arm pushes overlap
still lower.


In [3]:
# this cell plots within-corpus lexical overlap (R4) plus unique-opening counts
r4 = e11["r4_diversity_EXPLORATORY"]
names = ["selfgen", "weak_external", "strong_external"]
labels = ["self-generated\n(gemma-it)", "weak external\n(gemma-4-4B corpus)", "fixed DeepSeek"]
jac = [r4[n]["mean_pairwise_jaccard"] if r4.get(n) else None for n in names]
fig = go.Figure(go.Bar(x=labels, y=jac, text=[f"{v:.4f}" if v else "n/a" for v in jac], textposition="outside"))
fig.update_layout(
    title=f"Within-corpus 5-gram Jaccard overlap (higher = more repetitive) | probes read from {MODEL}",
    yaxis_title="mean pairwise Jaccard", width=850, height=420, margin=dict(t=80))
fig.show()


## 3. Detection accuracy per layer (E11's registered R1 read)

**How to read this:** each row is a probe lineage, each column a layer of
gemma-4-31b-it. Cell text is "paper battery / held-out battery" correct
counts out of 12 (target emotion in top 3 under centered cosine). A cell is
outlined when it clears the dual-battery bar (both counts at or above 8).
**What this buys:** the full picture behind the passing-layer headline, so
you can see WHERE each lineage works, not just how often. The diverse arm's
full-corpus grid comes from `e12_diverse_fullcorpus_grid.json`; the other
three rows are E11's frozen grid.


In [4]:
# this cell draws the per-layer dual-battery heatmap for all four probe sets
rows = {
    "self-generated n=256": e11["r1_dual_battery"]["selfgen"]["per_layer"],
    "weak external (4B corpus)": e11["r1_dual_battery"]["weak_external"]["per_layer"],
    "fixed DeepSeek n=256": e11["r1_dual_battery"]["strong_external"]["per_layer"],
    "diverse DeepSeek n=1024": full_grid["r1_dual_battery"]["diverse_deepseek_n1024"]["per_layer"],
}
layers = sorted(int(k) for k in next(iter(rows.values())))
z, text = [], []
for per_layer in rows.values():
    counts = [per_layer[str(L)] if str(L) in per_layer else per_layer[L] for L in layers]
    z.append([min(p, h) for p, h in counts])
    text.append([f"{p}/{h}" + (" *" if p >= 8 and h >= 8 else "") for p, h in counts])
fig = go.Figure(go.Heatmap(
    z=z, x=[str(L) for L in layers], y=list(rows), text=text, texttemplate="%{text}",
    colorscale="Blues", zmin=0, zmax=12, colorbar_title="min(paper, held-out)"))
fig.update_layout(
    title=f"Dual-battery counts per layer (paper/held-out, * = passes the 8/12 bar) | {MODEL}",
    xaxis_title="layer", width=1150, height=430, margin=dict(t=80))
fig.show()
for name, r1_src in [("self-generated", e11["r1_dual_battery"]["selfgen"]),
                     ("weak external", e11["r1_dual_battery"]["weak_external"]),
                     ("fixed DeepSeek", e11["r1_dual_battery"]["strong_external"]),
                     ("diverse DeepSeek", full_grid["r1_dual_battery"]["diverse_deepseek_n1024"])]:
    print(f"{name:16s} passing layers: {r1_src['passing_layers']}")


self-generated   passing layers: [33, 39, 42, 51, 54]
weak external    passing layers: [42]
fixed DeepSeek   passing layers: [6, 12, 33, 36, 39, 42, 45, 54, 57]
diverse DeepSeek passing layers: [6, 9, 12, 36, 39, 42, 54]


## 4. Dose-response: how many stories do probes actually need? (E12)

**How to read this:** x is stories per emotion (log scale), y is how many of
the 20 layers pass the dual-battery bar. Markers are 5 seeded subsamples per
n; the line tracks the seed mean; the star is the full corpus. **What this
buys:** the answer to "would more stories help?". The fixed arm saturates
near n=64; horizontal reference lines mark self-generated probes (5 passing
layers at n=256) and the fixed-corpus ceiling (9). The right panel shows the
contrast cosine of each subsample's probes to the self-generated probes at
layer 33: probe FUNCTION saturates before probe GEOMETRY stops moving.


In [5]:
# this cell plots passing-layer count and probe-direction convergence versus n
src = both_curves["arms"] if both_curves else {"fixed": fixed_curve["arms"]["fixed"]}
fig = make_subplots(rows=1, cols=2, subplot_titles=(
    "dual-battery passing layers vs corpus size",
    "probe direction cosine to self-generated probes (layer 33)"))
colors = {"fixed": "#1f77b4", "diverse": "#d62728"}
for arm, curve in src.items():
    pts = curve["points"]
    ns = sorted({p["n"] for p in pts})
    for col, key in ((1, "n_passing_layers"), (2, "mean_contrast_cos_to_selfgen_L33")):
        mean = [np.mean([p[key] for p in pts if p["n"] == n]) for n in ns]
        fig.add_scatter(x=[p["n"] for p in pts], y=[p[key] for p in pts], mode="markers",
                        marker=dict(color=colors.get(arm, "gray"), opacity=0.4),
                        name=f"{arm} seeds", legendgroup=arm, showlegend=(col == 1), row=1, col=col)
        fig.add_scatter(x=ns, y=mean, mode="lines+markers",
                        line=dict(color=colors.get(arm, "gray")),
                        name=f"{arm} mean", legendgroup=arm, showlegend=(col == 1), row=1, col=col)
fig.add_hline(y=9, line_dash="dot", annotation_text="fixed-corpus ceiling (9)", row=1, col=1)
fig.add_hline(y=5, line_dash="dot", annotation_text="self-generated n=256 (5)", row=1, col=1)
fig.update_xaxes(type="log", title="stories per emotion")
fig.update_yaxes(title="passing layers (of 20)", row=1, col=1)
fig.update_yaxes(title="mean contrast cosine", row=1, col=2)
fig.update_layout(title=f"E12 dose-response, 5 seeds per point | probes read from {MODEL}",
                  width=1150, height=470, margin=dict(t=90))
fig.show()


## 5. Direction geometry across lineages (E11's R2 read)

**How to read this:** each cell compares two probe sets at layer 33
(geometry peak). Top number: mean per-emotion contrast cosine (do the
DIRECTIONS agree?). Bottom number: RSA over the 12x12 emotion-similarity
matrices (does the SHAPE of emotion space agree?). **What this buys:** the
mechanism picture. A stronger generator converges toward the probed model's
own directions (0.57 versus the weak corpus's 0.22), and the diverse arm
keeps the shape (RSA 0.70) while rotating individual directions (cos 0.46).


In [6]:
# this cell draws the pairwise cosine and RSA matrix across the four lineages
pairs = {**e11["r2_cross_lineage_layer33"], **full_grid["r2_cross_lineage_layer33"]}
short = {"selfgen": "self-gen", "selfgen_postfix": "self-gen",
         "weak_external": "weak ext", "fixed_deepseek_n256": "fixed DS",
         "strong_external": "fixed DS", "diverse_deepseek_n1024": "diverse DS"}
names = ["self-gen", "weak ext", "fixed DS", "diverse DS"]
cos = np.full((4, 4), np.nan); rsa = np.full((4, 4), np.nan)
for k, v in pairs.items():
    a, b = k.split("_vs_")
    if short.get(a) in names and short.get(b) in names:
        i, j = names.index(short[a]), names.index(short[b])
        cos[i, j] = cos[j, i] = v["mean_contrast_cos"]
        rsa[i, j] = rsa[j, i] = v["rsa_12x12"]
text = [[("" if np.isnan(cos[i][j]) else f"cos {cos[i][j]:.2f}<br>RSA {rsa[i][j]:.2f}")
         for j in range(4)] for i in range(4)]
fig = go.Figure(go.Heatmap(z=cos, x=names, y=names, text=text, texttemplate="%{text}",
                           colorscale="RdBu", zmid=0, colorbar_title="contrast cos"))
fig.update_layout(title=f"Cross-lineage probe agreement at layer 33 | {MODEL}",
                  width=750, height=560, margin=dict(t=80))
fig.show()


## 6. The preference read: where corpus quality still pays (R3)

**How to read this:** for 64 activities with measured preference Elo scores
(fixed chat instrument, post-fix), the max absolute correlation between any
probe's cosine and Elo, per lineage. **What this buys:** a finer-grained
functional read than top-3 detection. Detection saturates (section 4), but
this read keeps improving with corpus quality and diversity, self-generated
0.62 to fixed DeepSeek 0.71 to diverse DeepSeek 0.77. Caveat: max over 12
emotions x 4 layers per bar, so treat levels, not tiny gaps, as the signal.


In [7]:
# this cell plots the preference probe-Elo correlation per lineage
r3 = {"self-generated": e11["r3_preference_probe_elo"]["selfgen"],
      "weak external": e11["r3_preference_probe_elo"]["weak_external"],
      "fixed DeepSeek": full_grid["r3_preference_probe_elo"]["fixed_deepseek_n256"],
      "diverse DeepSeek": full_grid["r3_preference_probe_elo"]["diverse_deepseek_n1024"]}
fig = go.Figure(go.Bar(
    x=list(r3), y=[v["abs_r"] for v in r3.values()],
    text=[f"{v['abs_r']:.3f}<br>L{v['layer']} {v['emotion']}" for v in r3.values()],
    textposition="outside"))
fig.update_layout(
    title=f"Max |r| between probe cosine and preference Elo (64 activities) | {MODEL}",
    yaxis_title="max |Pearson r|", yaxis_range=[0, 0.9], width=850, height=440, margin=dict(t=80))
fig.show()


## 7. What we are saying, exactly

1. **Generator quality, not generator identity, drives probe function**
   (E11, gated by C4's falsify pass): fixed DeepSeek probes pass at 9 layers
   versus self-generated 5 and weak-external 1.
2. **Detection probes saturate early** (E12): the fixed arm reaches its
   ceiling near 64 stories per emotion; 64 strong-generator stories already
   beat 256 self-generated ones.
3. **Diversity and scale do not raise the detection ceiling** (E12): 1024
   diverse stories pass at 7 layers, below the fixed corpus's 9. The
   registered growth branch did not fire.
4. **The preference read still rewards better corpora**: 0.62 (self-gen) to
   0.71 (fixed DeepSeek) to 0.77 (diverse DeepSeek). Coarse detection
   saturates; the finer behavioral correlate does not.
5. **Practical recipe**: any strong generator, roughly 64-256 fixed-prompt
   stories per emotion, is enough for detection probes; invest in corpus
   quality and diversity only when the downstream read is fine-grained.

**Caveats.** One strong external generator tested (DeepSeek-v4-pro);
"quality" is operationalized by that single point plus the weak 4B corpus.
The diverse arm changed the prompt, not just diversity (persona and setting
content are shared across emotions and removed by 12-pool centering, but
second-order interactions are not controlled). The preference ordering is a
three-point trend on a max-statistic, not a gated claim; matched-n reads
come from `e12_scale_curve.json` when both arms are present.
